In [9]:
!pip install uaibot

In [10]:
"""Smooth box SDF and batched box-to-oriented-point-cloud h-NESDF."""

import torch


"""Smooth box SDF and batched box-to-oriented-point-cloud h-NESDF."""

import torch


def h_esdf_box(T, lx, ly, lz, p, h):
    """Box-to-point field; T (...,4,4), p (...,3), broadcastable.

    Dimensions are full side lengths; h and dimensions must be positive.
    T and p must be floating tensors on the same device with the same dtype.
    For B poses and K points, pass T[:, None] and p[None].
    """
    u = (T[..., :3, :3].transpose(-1, -2)
         @ (p - T[..., :3, 3]).unsqueeze(-1)).squeeze(-1)
    a = p.new_tensor([lx, ly, lz]) / 2
    q = torch.logsumexp(torch.stack((u / h, -u / h), -1), -1) - a / h
    positive = torch.logsumexp(torch.stack((q, torch.zeros_like(q)), -1), -1)
    maximum = torch.logsumexp(q, -1)
    negative = torch.logsumexp(
        torch.stack((-maximum, torch.zeros_like(maximum)), -1), -1)
    # Norm radius h**2 / ||a|| vanishes faster than the LSE bandwidth h.
    outside = torch.sqrt(positive.square().sum(-1) + (h / a.norm()).square())
    return h * (outside - negative)


@torch.enable_grad()
def hNESDF(batch_T, lx, ly, lz, h, list_points, *,
           return_tensors=False, compute_gradient=True,
           gradient_method="autograd", fd_steps=(1e-3, 1e-3)):
    """Return distance list and list of 6D pose differentials.

    batch_T: (B,4,4) floating tensor of box poses.
    list_points: (K,6) tensor [px,py,pz,nx,ny,nz], shared by all poses.
        Points are approximately uniform boundary samples of the other body;
        normals are outward unit normals. Both are in world coordinates.
    Each differential L uses xi=[vx,vy,vz,wx,wy,wz], with world-frame
    origin velocity s_dot=v and angular velocity R_dot=[w]_x R.
    Set return_tensors=True to keep detached outputs on the input device.
    Set compute_gradient=False for distance-only evaluation; L is then None.
    gradient_method: "autograd" or "finite_difference" for the pose gradient.
    fd_steps: (translation step in length units, rotation step in radians).
        Central differences evaluate perturbations sequentially. This avoids
        second-order autograd; the spatial derivative still uses autograd.
    """
    if batch_T.ndim != 3 or batch_T.shape[1:] != (4, 4):
        raise ValueError("batch_T must have shape (B,4,4).")
    if gradient_method not in ("autograd", "finite_difference"):
        raise ValueError("gradient_method must be 'autograd' or 'finite_difference'.")
    if compute_gradient and gradient_method == "finite_difference":
        if len(fd_steps) != 2 or not all(0 < x < float("inf") for x in fd_steps):
            raise ValueError("fd_steps must contain two positive steps.")
        if batch_T.dtype not in (torch.float32, torch.float64):
            raise ValueError("Use float32 or float64 for finite differences.")
        T = batch_T.detach()
        distance, _ = hNESDF(T, lx, ly, lz, h, list_points,
                            return_tensors=True, compute_gradient=False)
        L = T.new_empty((len(T), 6))
        for i in range(6):
            eps = fd_steps[i // 3]
            values = []
            for sign in (1, -1):
                perturbed = T.clone()
                if i < 3:
                    perturbed[:, i, 3] += sign * eps
                else:
                    W = T.new_zeros((3, 3))
                    a, b = (i - 2) % 3, (i - 1) % 3
                    W[a, b], W[b, a] = -sign * eps, sign * eps
                    perturbed[:, :3, :3] = torch.matrix_exp(W) @ T[:, :3, :3]
                value, _ = hNESDF(perturbed, lx, ly, lz, h, list_points,
                                 return_tensors=True, compute_gradient=False)
                values.append(value)
            L[:, i] = (values[0] - values[1]) / (2 * eps)
        if return_tensors:
            return distance, L
        return distance.cpu().tolist(), L.cpu().tolist()
    T = batch_T.detach().clone().requires_grad_(compute_gradient)
    cloud = torch.as_tensor(list_points, dtype=T.dtype, device=T.device)
    if cloud.ndim != 2 or cloud.shape[1] != 6 or cloud.shape[0] == 0:
        raise ValueError("list_points must be a nonempty (K,6) oriented cloud.")
    p = cloud[None, :, :3].expand(T.shape[0], -1, -1)
    p = p.detach().clone().requires_grad_(True)

    D = h_esdf_box(T[:, None], lx, ly, lz, p, h)
    d = h_esdf_box(T, lx, ly, lz, T[:, :3, 3], h)
    E = 0.5 * (D - d[:, None]).square()  # Constant -d**2/2 cancels.
    grad_E = torch.autograd.grad(E.sum(), p, create_graph=compute_gradient)[0]
    if not compute_gradient:
        E = E.detach()  # No pose derivative is needed after the spatial derivative.
    weights = torch.softmax(-E / h**2, dim=-1)
    projection = (grad_E * cloud[None, :, 3:]).sum(-1)
    distance = d - (weights * projection).sum(-1)

    L = None
    if compute_gradient:
        G = torch.autograd.grad(distance.sum(), T)[0]
        M = G[:, :3, :3] @ T[:, :3, :3].transpose(-1, -2)
        angular = torch.stack((M[:, 2, 1] - M[:, 1, 2],
                               M[:, 0, 2] - M[:, 2, 0],
                               M[:, 1, 0] - M[:, 0, 1]), -1)
        L = torch.cat((G[:, :3, 3], angular), -1).detach()
    distance = distance.detach()
    if return_tensors:
        return distance, L
    return distance.cpu().tolist(), None if L is None else L.cpu().tolist()


import numpy as np
import torch


def sample_box_environment(scenarios, delta, *, device=None,
                           dtype=torch.float32, tol=1e-9):
    """
    scenarios: list of (T, lx, ly, lz), with full side lengths.
    delta: maximum grid spacing along each face direction.
    tol: geometric tolerance, in the same units as the lengths.

    Returns a (K,6) tensor of world-frame points and outward unit normals.
    """
    if not np.isfinite(delta) or delta <= 0 or not np.isfinite(tol) or tol < 0:
        raise ValueError("delta must be positive and tol nonnegative.")

    boxes = []
    for T, lx, ly, lz in scenarios:
        T = np.asarray(T, dtype=float)  # Also accepts np.matrix.
        lengths = np.array([lx, ly, lz], dtype=float)
        if (T.shape != (4, 4) or not np.isfinite(T).all()
                or not np.isfinite(lengths).all() or np.any(lengths <= 0)):
            raise ValueError("Each box needs an SE(3) pose and positive lengths.")
        boxes.append((T[:3, :3], T[:3, 3], lengths / 2))

    samples = []
    for i, (R, center, half) in enumerate(boxes):
        for axis in range(3):
            tangents = [k for k in range(3) if k != axis]
            count = np.ceil(2 * half[tangents] / delta).astype(int)

            # Cell centers avoid sampling edges with ambiguous normals.
            grids = [
                (np.arange(n) + 0.5) * (2 * half[k] / n) - half[k]
                for k, n in zip(tangents, count)
            ]
            grid = np.stack(
                np.meshgrid(*grids, indexing="ij"), -1
            ).reshape(-1, 2)

            for side in (-1, 1):
                local = np.zeros((len(grid), 3))
                local[:, tangents] = grid
                local[:, axis] = side * half[axis]

                points = local @ R.T + center
                normal = side * R[:, axis]
                keep = np.ones(len(points), dtype=bool)

                for j, (Rj, cj, hj) in enumerate(boxes):
                    if j == i or not keep.any():
                        continue

                    ids = np.flatnonzero(keep)
                    q = (points[ids] - cj) @ Rj
                    gap = hj - np.abs(q)

                    inside = np.all(gap > tol, axis=1)
                    closed = np.all(gap >= -tol, axis=1)
                    boundary = closed & ~inside

                    # Check whether a coincident face points outward
                    # in the same direction as the sampled face.
                    direction = normal @ Rj
                    same_normal = np.any(
                        (np.abs(gap) <= tol)
                        & (np.sign(q) * direction > 1 - 1e-10),
                        axis=1,
                    )

                    # Remove buried/glued points. Shared exposed patches
                    # belong to the first box, preventing double density.
                    hidden = inside | (
                        boundary & (~same_normal | (j < i))
                    )
                    keep[ids[hidden]] = False

                points = points[keep]
                if len(points):
                    normals = np.broadcast_to(normal, points.shape)
                    samples.append(np.concatenate((points, normals), axis=1))

    cloud = np.concatenate(samples) if samples else np.empty((0, 6))
    return torch.as_tensor(cloud, dtype=dtype, device=device)

"""A furnished 12 m x 10 m house made entirely of box tuples."""

import numpy as np


def create_house_environment(include_ceiling=False):
    """Return [(T, lx, ly, lz), ...], with 4x4 np.matrix poses.

    Units are metres. The floor's upper surface is z=0. The house occupies
    x in [-6,6], y in [-5,5], with 2.8 m high walls. Its 1.6 m wide central
    corridor connects a 1.2 m entrance to four 1.0 m room doorways; all
    doorway clear heights are 2.1 m. Four side windows are 1.6 x 1.0 m.
    Doorways/windows are gaps in the box geometry, with no door leaves/glass.
    Set include_ceiling=True to add a ceiling at z=2.8 m.
    """
    boxes = []

    def add(center, size, yaw=0.0):
        c, s = np.cos(yaw), np.sin(yaw)
        T = np.eye(4)
        T[:3, :3] = [[c, -s, 0], [s, c, 0], [0, 0, 1]]
        T[:3, 3] = center
        boxes.append((np.matrix(T), *map(float, size)))

    # Floor, with its top at z=0.
    add((0, 0, -0.10), (12, 10, 0.20))

    # West/east walls. Sills and lintels leave windows at y=-2.0 and 2.2,
    # with clear z range [1.0,2.0].
    for x in (-5.9, 5.9):
        add((x, 0, 0.5), (0.2, 9.6, 1.0))
        add((x, 0, 2.4), (0.2, 9.6, 0.8))
        for lo, hi in [(-4.8, -2.8), (-1.2, 1.4), (3.0, 4.8)]:
            add((x, (lo + hi) / 2, 1.5), (0.2, hi - lo, 1.0))

    # North wall and south entrance: x in [-0.6,0.6] remains open below 2.1.
    add((0, 4.9, 1.4), (12, 0.2, 2.8))
    for x in (-3.3, 3.3):
        add((x, -4.9, 1.4), (5.4, 0.2, 2.8))
    add((0, -4.9, 2.45), (1.2, 0.2, 0.7))

    # Corridor: clear x range [-0.8,0.8]. Room doors at y=-2.0 and 2.2.
    for x in (-0.9, 0.9):
        for lo, hi in [(-4.8, -2.5), (-1.5, 1.7), (2.7, 4.8)]:
            add((x, (lo + hi) / 2, 1.4), (0.2, hi - lo, 2.8))
        for y in (-2.0, 2.2):
            add((x, y, 2.45), (0.2, 1.0, 0.7))

    # Separate the front/back rooms while leaving the corridor continuous.
    for x in (-3.4, 3.4):
        add((x, 0.1, 1.4), (4.8, 0.2, 2.8))

    # Southwest: dining table with four legs.
    add((-3.4, -2.2, 0.76), (1.5, 0.9, 0.08))
    for dx in (-0.62, 0.62):
        for dy in (-0.33, 0.33):
            add((-3.4 + dx, -2.2 + dy, 0.36), (0.08, 0.08, 0.72))

    # Sofa: base/seat, back and two armrests.
    add((-5.25, -1.45, 0.30), (0.9, 2.2, 0.6))
    add((-5.65, -1.45, 0.80), (0.12, 2.2, 0.4))
    for y in (-2.50, -0.40):
        add((-5.25, y, 0.75), (0.9, 0.12, 0.3))

    # TV cabinet, pedestal and screen facing the sofa.
    add((-1.45, -0.65, 0.25), (0.65, 1.1, 0.5))
    add((-1.45, -0.65, 0.55), (0.08, 0.2, 0.1))
    add((-1.45, -0.65, 0.95), (0.04, 0.95, 0.7))

    # Four dining chairs and one slightly rotated desk chair.
    # Each chair comprises a seat, a backrest and four legs.
    parts = [((0, 0, 0.45), (0.44, 0.44, 0.08)),
             ((0, 0.20, 0.74), (0.44, 0.04, 0.50))]
    parts += [((dx, dy, 0.205), (0.045, 0.045, 0.41))
              for dx in (-0.16, 0.16) for dy in (-0.16, 0.16)]
    chairs = [(-3.9, -3.0, np.pi), (-2.9, -3.0, np.pi),
              (-3.9, -1.35, 0), (-2.9, -1.35, 0),
              (4.2, 2.75, np.pi + np.pi / 12)]
    for x, y, yaw in chairs:
        c, s = np.cos(yaw), np.sin(yaw)
        for (dx, dy, z), size in parts:
            add((x + c * dx - s * dy, y + s * dx + c * dy, z), size, yaw)

    # Northwest: bed frame, mattress, headboard, two pillows and storage.
    add((-3.2, 3.3, 0.20), (1.8, 2.2, 0.4))
    add((-3.2, 3.3, 0.50), (1.7, 2.1, 0.2))
    add((-3.2, 4.45, 0.70), (1.9, 0.12, 1.4))
    for x in (-3.65, -2.85):
        add((x, 4.0, 0.65), (0.55, 0.4, 0.1))
    add((-5.2, 3.7, 1.0), (1.0, 1.5, 2.0))
    add((-1.65, 4.0, 0.30), (0.55, 0.55, 0.6))

    # Southeast: L-shaped kitchen counter, island and refrigerator.
    add((3.4, -4.45, 0.45), (4.2, 0.7, 0.9))
    add((5.45, -2.75, 0.45), (0.7, 2.7, 0.9))
    add((3.3, -2.4, 0.45), (1.1, 1.4, 0.9))
    add((5.25, -0.55, 0.95), (1.0, 0.9, 1.9))

    # Northeast: desk with legs, and an open bookcase with four shelves.
    add((4.2, 3.55, 0.78), (1.8, 0.8, 0.08))
    for dx in (-0.76, 0.76):
        for dy in (-0.26, 0.26):
            add((4.2 + dx, 3.55 + dy, 0.37), (0.08, 0.08, 0.74))
    for x in (1.44, 2.96):
        add((x, 4.45, 0.9), (0.08, 0.4, 1.8))
    for z in (0.04, 0.60, 1.16, 1.76):
        add((2.2, 4.45, z), (1.44, 0.4, 0.08))
    add((2.2, 4.61, 0.9), (1.44, 0.08, 1.8))

    if include_ceiling:
        add((0, 0, 2.9), (12, 10, 0.2))

    return boxes

In [11]:
"""Batched h-NESDF path optimization with Nesterov-style SE(3) updates."""

import numpy as np
import torch
from scipy.spatial.transform import Rotation, Slerp



def _interpolate(poses, subdivisions):
    """Linear translation and shortest-arc SLERP on each segment."""
    poses = np.asarray(poses)
    times = np.arange(len(poses), dtype=float)
    query = np.linspace(0, times[-1], (len(poses) - 1) * subdivisions + 1)
    result = np.repeat(np.eye(4)[None], len(query), axis=0)
    result[:, :3, :3] = Slerp(
        times, Rotation.from_matrix(poses[:, :3, :3])
    )(query).as_matrix()
    for j in range(3):
        result[:, j, 3] = np.interp(query, times, poses[:, j, 3])
    result[[0, -1]] = poses[[0, -1]]
    return result


@torch.no_grad()
def _move(H, increments):
    """World-frame increments [dt,domega], with t_dot=v, R_dot=[w] R."""
    w = increments[:, 3:]
    W = H.new_zeros((len(H), 3, 3))
    W[:, 0, 1], W[:, 1, 0] = -w[:, 2], w[:, 2]
    W[:, 0, 2], W[:, 2, 0] = w[:, 1], -w[:, 1]
    W[:, 1, 2], W[:, 2, 1] = -w[:, 0], w[:, 0]
    result = H.clone()
    result[:, :3, :3] = torch.matrix_exp(W) @ H[:, :3, :3]
    result[:, :3, 3] += increments[:, :3]
    result[[0, -1]] = H[[0, -1]]
    return result


@torch.enable_grad()
def path_objective(H, lx, ly, lz, h, cloud, threshold, G=10.0,
                   softmin_h=None, rotation_scale=1.0, batch_size=16,
                   gradient=True, gradient_method="autograd",
                   fd_steps=(1e-3, 1e-3), proximity_gain=10.0,
                   proximity_epsilon=0.05):
    """Return (objective, distances, world-frame gradient, max_step).

    Invalid spacing returns (-inf, None, None, max_step). The gradient
    has shape (N+1,6), with fixed-endpoint rows zero. Tensor outputs are
    detached and remain on H.device; objective/max_step are Python floats.
    CHOMP's proximity penalty is integrated along pose arc length with
    trapezoidal weights. G retains the original soft-min clearance reward.
    """
    if (not np.isfinite(proximity_gain) or proximity_gain < 0
            or not np.isfinite(proximity_epsilon) or proximity_epsilon <= 0):
        raise ValueError("proximity_gain must be nonnegative and proximity_epsilon positive.")
    tau = h if softmin_h is None else softmin_h
    X = H.detach().clone().requires_grad_(gradient)
    dt = X[1:, :3, 3] - X[:-1, :3, 3]
    dR = X[1:, :3, :3] - X[:-1, :3, :3]
    s = dt.square().sum(-1) + rotation_scale**2 * dR.square().sum((-1, -2))
    max_step = float(s.detach().max().sqrt())
    if max_step >= threshold:
        return -float("inf"), None, None, max_step
    reward = torch.log1p(-s / threshold**2).sum()

    distances, differentials = [], []
    chunk = len(X) if batch_size is None else batch_size
    for part in X.split(chunk):
        d, L = hNESDF(part, lx, ly, lz, h, cloud,
                      return_tensors=True, compute_gradient=gradient,
                      gradient_method=gradient_method, fd_steps=fd_steps)
        distances.append(d)
        if gradient:
            differentials.append(L)
    d = torch.cat(distances)
    base = reward
    distance_weight = G * torch.softmax(-d / tau, dim=0)
    if proximity_gain > 0:
        eps = proximity_epsilon
        rho = torch.where(d < 0, -d + eps / 2,
                          (eps - d).clamp_min(0).square() / (2 * eps))
        # This norm has a finite zero subgradient at a repeated waypoint.
        length = torch.linalg.vector_norm(
            torch.cat((dt, rotation_scale * dR.flatten(1)), dim=1), dim=1)
        base = base - proximity_gain * (0.5 * (rho[:-1] + rho[1:]) * length).sum()
        # Each waypoint receives half the arc length of its adjacent edges.
        weights = d.new_zeros(d.shape)
        weights[:-1] += length.detach() / 2
        weights[1:] += length.detach() / 2
        minus_rho_prime = (1 - d / eps).clamp(0, 1)
        distance_weight += proximity_gain * weights * minus_rho_prime
    f = base - G * tau * torch.logsumexp(-d / tau, dim=0)

    grad = None
    if gradient:
        A = torch.autograd.grad(base, X)[0]
        M = A[:, :3, :3] @ X[:, :3, :3].transpose(-1, -2)
        angular = torch.stack((M[:, 2, 1] - M[:, 1, 2],
                               M[:, 0, 2] - M[:, 2, 0],
                               M[:, 1, 0] - M[:, 0, 1]), -1)
        grad = torch.cat((A[:, :3, 3], angular), -1).detach()
        grad += distance_weight[:, None] * torch.cat(differentials)
        grad[[0, -1]] = 0
    return float(f.detach()), d, grad, max_step


def chomp_se3(H_start, H_goal, lx, ly, lz, h, list_points, N=40, *,
              threshold=0.4, G=10.0, iterations=300, step_size=0.05,
              momentum=0.8, clearance=0.05, softmin_h=None,
              rotation_scale=1.0, batch_size=16, save_every=1,
              check_substeps=5, initial_path=None, precondition=True,
              stop_when_clear=True, gradient_method="autograd",
              fd_steps=(1e-3, 1e-3), proximity_gain=10.0,
              proximity_epsilon=None, echo = True):
    """Return (paths, info); paths[-1] is the final waypoint sequence.

    N is the number of segments: there are N+1 poses and N-1 free poses.
    Each saved path is a NumPy array (N+1,4,4), including fixed endpoints.
    Default initialization is linear translation plus shortest-arc SLERP.
    initial_path, when given, must have that same shape and endpoint poses.

    The maximized objective is sum(log(1-s_k/threshold**2))
    -G*tau*logsumexp(-d_k/tau) - proximity_gain*C, with tau=softmin_h or h and
    s_k=||t[k+1]-t[k]||^2+rotation_scale^2*||R[k+1]-R[k]||_F^2.
    C=sum_k sqrt(s_k)*(rho(d_k)+rho(d_{k+1}))/2, where rho(d) is CHOMP's
    penalty: -d+epsilon/2 for d<0, (d-epsilon)**2/(2*epsilon) for
    0<=d<=epsilon, and zero otherwise. epsilon=proximity_epsilon defaults
    to clearance when positive, or h otherwise. This is a pose-arc-length
    adaptation, using the whole-box h-NESDF in place of CHOMP's body-point
    workspace integral. Its gradient includes the arc-length derivative.
    Set proximity_gain=0 to recover the previous objective, or G=0 to use
    only the CHOMP proximity penalty plus the original spacing barrier.
    rotation_scale is a length scale; 1 reproduces the numerical matrix norm.
    Backtracking maintains the strict spacing bound and monotone objective.
    Optional temporal-Laplacian preconditioning spreads updates along the path.

    info['success'] means a DENSE SAMPLED h-NESDF margin check passed.
    It is NOT an exact or continuous swept-volume collision certificate.
    info['dense_path'] uses the same linear/SLERP interpolation as initialization.
    This is a local optimizer and can fail from a poor initial path.
    stop_when_clear=False continues optimizing after waypoint clearance
    is reached, up to iterations or a line-search stall.
    gradient_method="finite_difference" uses central differences only for
    the h-NESDF pose gradient; the spacing gradient remains exact.
    fd_steps=(translation step, rotation step in radians). Keep float32 or
    float64 computation; tune the steps to the scene scale and bandwidth.
    """
    tau = h if softmin_h is None else softmin_h
    eps = (clearance if clearance > 0 else h) if proximity_epsilon is None else proximity_epsilon
    if (N < 2 or int(N) != N or min(lx, ly, lz, h, tau, threshold,
                                    step_size, rotation_scale) <= 0
            or G < 0 or not 0 <= momentum < 1 or iterations < 0
            or save_every < 1 or check_substeps < 1
            or (batch_size is not None and batch_size < 1)):
        raise ValueError("Invalid dimensions, bandwidth, or optimization parameters.")
    if gradient_method not in ("autograd", "finite_difference"):
        raise ValueError("Unknown gradient_method.")
    if (not np.isfinite(proximity_gain) or proximity_gain < 0
            or not np.isfinite(eps) or eps <= 0):
        raise ValueError("proximity_gain must be nonnegative and proximity_epsilon positive.")
    N = int(N)
    cloud = torch.as_tensor(list_points).detach()
    if not cloud.is_floating_point():
        cloud = cloud.float()
    start = torch.as_tensor(H_start, dtype=cloud.dtype, device=cloud.device)
    goal = torch.as_tensor(H_goal, dtype=cloud.dtype, device=cloud.device)
    endpoints = torch.stack((start, goal)).detach().cpu().numpy()
    if isinstance(initial_path, torch.Tensor):
        initial_path = initial_path.detach().cpu().numpy()
    seed = _interpolate(endpoints, N) if initial_path is None else np.asarray(initial_path)
    if seed.shape != (N + 1, 4, 4):
        raise ValueError("initial_path must have shape (N+1,4,4).")
    H = torch.as_tensor(seed.copy(), dtype=cloud.dtype, device=cloud.device)
    H[[0, -1]] = torch.stack((start, goal))
    velocity = torch.zeros((N + 1, 6), dtype=H.dtype, device=H.device)

    # A positive-definite discrete Laplacian on the free waypoints.
    n = N - 1
    A = 2 * torch.eye(n, dtype=H.dtype, device=H.device)
    A += torch.diag(-torch.ones(n - 1, dtype=H.dtype, device=H.device), 1)
    A += torch.diag(-torch.ones(n - 1, dtype=H.dtype, device=H.device), -1)
    chol = torch.linalg.cholesky(A) if precondition else None

    def evaluate(X, gradient):
        return path_objective(X, lx, ly, lz, h, cloud, threshold, G,
                              tau, rotation_scale, batch_size, gradient,
                              gradient_method, fd_steps, proximity_gain, eps)

    f, d, _, max_step = evaluate(H, False)
    if d is None:
        raise ValueError(f"Initial max step is {max_step:.6g}; increase N or threshold.")
    if float(d[[0, -1]].min()) < clearance:
        raise ValueError("A fixed endpoint is below the requested h-NESDF clearance.")
    paths = [H.detach().cpu().numpy().copy()]
    saved_iterations = [0]
    objectives, minimums, maximums = [f], [float(d.min())], [max_step]
    status, completed, next_step = "max_iterations", 0, step_size

    for iteration in range(1, iterations + 1):
        if echo:
          print("i = "+str(iteration)+", d_min = "+str(float(d.min())))

        if stop_when_clear and float(d.min()) >= clearance:
            status = "waypoint_clearance_reached"
            break
        accepted = False
        for restart in range(2):
            inertia = momentum * velocity if restart == 0 else torch.zeros_like(velocity)
            Y = _move(H, inertia)
            fy, _, grad, _ = evaluate(Y, True)
            if grad is None or not np.isfinite(fy) or fy < f:
                continue
            direction = grad.clone()
            if precondition:
                direction[1:-1] = torch.cholesky_solve(direction[1:-1], chol)
            direction[:, 3:] /= rotation_scale**2
            norm = (direction[:, :3].square().sum(-1)
                    + rotation_scale**2 * direction[:, 3:].square().sum(-1)).sqrt().max()
            direction /= norm.clamp_min(1.0)
            slope = float((grad * direction).sum())
            if not np.isfinite(slope) or slope <= 1e-14:
                continue
            alpha = next_step
            for _ in range(24):
                increments = inertia + alpha * direction
                trial = _move(H, increments)
                ft, dt, _, mt = evaluate(trial, False)
                if np.isfinite(ft) and ft >= fy + 1e-4 * alpha * slope:
                    H, velocity, f, d, max_step = trial, increments, ft, dt, mt
                    next_step = min(step_size, 1.2 * alpha)
                    accepted = True
                    break
                alpha *= 0.5
            if accepted:
                break
        if not accepted:
            status = "stalled"
            break
        completed = iteration
        objectives.append(f)
        minimums.append(float(d.min()))
        maximums.append(max_step)
        if iteration % save_every == 0:
            paths.append(H.detach().cpu().numpy().copy())
            saved_iterations.append(iteration)

    if saved_iterations[-1] != completed:
        paths.append(H.detach().cpu().numpy().copy())
        saved_iterations.append(completed)
    final = H.detach().cpu().numpy().copy()
    dense_path = _interpolate(final, check_substeps)
    dense_T = torch.as_tensor(dense_path, dtype=H.dtype, device=H.device)
    chunk = len(dense_T) if batch_size is None else batch_size
    dense_distances = torch.cat([
        hNESDF(part, lx, ly, lz, h, cloud,
               return_tensors=True, compute_gradient=False)[0]
        for part in dense_T.split(chunk)
    ]).cpu().numpy()
    success = bool(np.isfinite(dense_distances).all()
                   and dense_distances.min() >= clearance)
    if success:
        status = "dense_sampled_clearance_reached"
    elif float(d.min()) >= clearance:
        status = "dense_check_failed"
    info = dict(success=success, status=status, iterations=completed,
                saved_iterations=saved_iterations, objective_history=objectives,
                min_distance_history=minimums, max_step_history=maximums,
                final_distances=d.cpu().numpy(), dense_path=dense_path,
                dense_distances=dense_distances,
                min_dense_distance=float(dense_distances.min()),
                clearance=clearance, threshold=threshold,
                proximity_gain=proximity_gain, proximity_epsilon=eps)
    return paths, info

import numpy as np
from scipy.spatial.transform import Rotation, Slerp

def interpolate_poses(H_start, H_goal, N):
    """Return N intermediate poses as np.matrix, excluding endpoints."""
    if N == 0:
        return []

    A, B = np.asarray(H_start), np.asarray(H_goal)
    t = np.linspace(0, 1, N + 2)[1:-1]

    rotations = Rotation.from_matrix(
        np.stack([A[:3, :3], B[:3, :3]])
    )
    H = np.tile(np.eye(4), (N, 1, 1))
    H[:, :3, :3] = Slerp([0, 1], rotations)(t).as_matrix()
    H[:, :3, 3] = (
        (1 - t[:, None]) * A[:3, 3]
        + t[:, None] * B[:3, 3]
    )

    return [np.matrix(pose) for pose in H]

def generate_poses(_pc, _lx, _ly, _lz):
  box_test = ub.Box(width=_lx, depth=_ly, height=_lz)
  cont = True
  while cont:

    try:
      H_start = ub.Utils.htm_rand([-4.5,-4.5,0.5],[4.5,4.5,2.5],rot=np.pi/2)
      H_goal = ub.Utils.htm_rand([-4.5,-4.5,0.5],[4.5,4.5,2.5],rot=np.pi/2)



      cont = np.linalg.norm(H_start[:3,3]-H_goal[:3,3])<2.0

      if not cont:
        box_test.set_ani_frame(htm=H_start)
        _, _, dist, _ = box_test.compute_dist(_pc)
        cont = dist<0.3

      if not cont:
        box_test.set_ani_frame(htm=H_goal)
        _, _, dist, _ = box_test.compute_dist(_pc)
        cont = dist<0.3

      if not cont:
        poses = interpolate_poses(H_start, H_goal, 30)
        j = 0
        cont_internal = True
        while cont_internal:
          j=j+1
          box_test.set_ani_frame(htm=poses[j])
          _, _, dist, _ = box_test.compute_dist(_pc)
          cont_internal =  (dist >= 1e-4) and (j<30)

        cont = dist>= 1e-4

    except Exception as e:
      cont = True

  return H_start, H_goal

In [12]:
import uaibot as ub

# Check if CUDA is available, otherwise default to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
data = np.load("underwater_complex.npz", allow_pickle=False)
pc_points = torch.as_tensor(data["point_cloud"], dtype=torch.float32, device=device)
H_start, H_goal = data["H_start"], data["H_goal"]
lx, ly, lz = data["box_lengths"]

pc_points = pc_points[pc_points[:, 2] >= -0.2]

pc_numpy = np.matrix(pc_points.detach().cpu().numpy()[:, :3]).T

print(np.shape(pc_numpy))

pc = ub.PointCloud(points=pc_numpy, color='white', size=0.05)



#UAIBOT IMPLEMENTATION, JUST FOR VISUALIZATION


sim = ub.Simulation([],background_color="#009dc4")
sim.add(pc)

box_start = ub.Box(htm=H_start,width=lx,depth=ly,height=lz,color='cyan',opacity=0.7)
box_goal = ub.Box(htm=H_goal,width=lx,depth=ly,height=lz,color='magenta',opacity=0.7)
sim.add(box_start)
sim.add(box_goal)

box_trail = []

TRAIL_SIZE = 15

for j in range(TRAIL_SIZE):
  box_trail.append(ub.Box(width=lx,depth=ly,height=lz,color='pink',opacity=0.7))
  sim.add(box_trail[j])

sim.run()

FileNotFoundError: [Errno 2] No such file or directory: 'underwater_complex.npz'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:


import gc
import torch
from time import perf_counter

# Start the timer



# # Delete GPU tensors you no longer need.
# # Removing pc_points means you must recreate the point cloud.
# for name in ("batch_T", "distances", "gradients", "pc_points"):
#     globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()




results_info = []

no_success = 0

path_success = []
time_success = 0

for n in range(100):

  try:
    H_start, H_goal = generate_poses(pc, lx, ly, lz)

    start_time = perf_counter()

    paths, info = chomp_se3(
      H_start, H_goal,
      lx=lx, ly=ly, lz=lz,
      h=0.1, list_points=pc_points,
      N=30,
      threshold=0.6, #0.3 for N=60
      G=25.0,
      softmin_h=0.1,
      clearance=0.04,
      iterations=20,
      step_size=0.2,
      proximity_gain=0,
      proximity_epsilon=0.04,
      momentum=0.95, #0.95
      batch_size=30,  # Poses evaluated simultaneously; independent of N.
      save_every=1,
      check_substeps=10,
      gradient_method="autograd",
      fd_steps=(1e-3, 1e-3),
      echo = False
    )

    end_time = perf_counter()
    elapsed_time = end_time - start_time

    results_info.append([info['success'],elapsed_time])
    print(results_info[-1])
    if results_info[-1][0]:
      no_success+=1
      time_success+=results_info[-1][1]
      path_success = paths
      break


  except Exception as e:
    pass

print(no_success/30)


In [ ]:
from math import floor


sim = ub.Simulation([],background_color="#009dc4")
sim.add(pc)

box_start = ub.Box(htm=H_start,width=lx,depth=ly,height=lz,color='magenta',opacity=1)
box_goal = ub.Box(htm=H_goal,width=lx,depth=ly,height=lz,color='yellow',opacity=1)
sim.add(box_start)
sim.add(box_goal)

box_trail = []

TRAIL_SIZE = 15

for j in range(TRAIL_SIZE):
  box_trail.append(ub.Box(width=lx,depth=ly,height=lz,color='red',opacity=0.3))
  sim.add(box_trail[j])


box_start.add_ani_frame(0,H_start)
box_goal.add_ani_frame(0,H_goal)

tot_j = len(path_success[0])
for k in range(len(path_success)):
  for j in range(15):
    box_trail[j].add_ani_frame(k,path_success[k][round(j*(tot_j-4)/14)+3])

sim.run()

In [ ]:
print(info['iterations'])

In [ ]:
sim.run()

In [ ]:
sim.run()